In [211]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset
#from torch.nn import functional as F

import json
from pyarrow import parquet as pq

In [212]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            for row in rows:

                yield {
                    'input_ids' : torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[2]], dtype=torch.long),
                    'target_ids': torch.tensor(row[self.cols[3]], dtype=torch.long)
                }

def wrapper_collate_batch(pad_value_in, pad_value_out):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]
        target_list = [item['target_ids'] for item in batch]

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value_in)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value_out)
        target_padded = pad_sequence(target_list, batch_first=True, padding_value=pad_value_out)

        return (input_padded, output_padded, target_padded)

    return collate_batch


def load_tokenizer(path:str) -> tuple:

    with open(path) as f:
        tokenizer = json.load(f)
    
    index2word = {int(key): value for key, value in tokenizer["inverse_vocab"].items()}
    special_tokens = tokenizer["special_tokens"]

    return special_tokens, index2word


def index2word(input_tensor: torch.Tensor, vocab_mapper: dict, unk_token:int) -> list:

    indices_matrix = input_tensor.cpu().numpy()

    output = [
        [vocab_mapper.get(idx, vocab_mapper[unk_token]) for idx in sentence]
        for sentence in indices_matrix
    ]
    
    return output

In [213]:
sp_tokens_eng, vocab_idx2word_eng = load_tokenizer("../data/main/vocabs/eng_tokenizer.json")
sp_tokens_spa, vocab_idx2word_spa = load_tokenizer("../data/main/vocabs/spa_tokenizer.json")

pad_value_eng = sp_tokens_eng["[PAD]"]
pad_value_spa = sp_tokens_spa["[PAD]"]

In [214]:
train_path = "../data/main/spa-eng/train_spa_eng"
train_schema = pq.ParquetDataset(train_path)
train_cols = train_schema.schema.names

test_path = "../data/main/spa-eng/test_spa_eng"
test_schema = pq.ParquetDataset(test_path)
test_cols = test_schema.schema.names

In [215]:
train_data = LoadLanguageData(train_path, train_cols)
train_loader = DataLoader(train_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

test_data = LoadLanguageData(test_path, test_cols)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

In [216]:
import numpy as np

for eng_in, spa_out, spa_t in train_loader:
    break

print(np.array(index2word(eng_in, vocab_idx2word_eng, sp_tokens_eng["[UNK]"]))[:3], end="\n\n\n")
print(np.array(index2word(spa_out,vocab_idx2word_spa, sp_tokens_spa["[UNK]"]))[:3], end="\n\n\n")
print(np.array(index2word(spa_t,  vocab_idx2word_spa, sp_tokens_spa["[UNK]"]))[:3])

[['go' 'make' 'popcorn' '.' '[END]' '[PAD]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]']
 ['there' 'were' 'no' 'radios' 'in' 'those' 'days' '.' '[END]' '[PAD]'
  '[PAD]']
 ['there' "'s" 'a' 'small' 'pond' 'in' 'our' 'garden' '.' '[END]' '[PAD]']]


[['[START]' 'andá' 'a' 'hacer' 'pochoclo' '.' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['[START]' 'no' 'había' 'radios' 'aún' 'entonces' '.' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['[START]' 'hay' 'un' 'pequeño' 'estanque' 'en' 'nuestro' 'jardín' '.'
  '[PAD]' '[PAD]' '[PAD]']]


[['andá' 'a' 'hacer' 'pochoclo' '.' '[END]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['no' 'había' 'radios' 'aún' 'entonces' '.' '[END]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['hay' 'un' 'pequeño' 'estanque' 'en' 'nuestro' 'jardín' '.' '[END]'
  '[PAD]' '[PAD]' '[PAD]']]


In [217]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, layers, p_dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.dropout = nn.Dropout(p_dropout)

        self.gru = nn.GRU(embedding_size, hidden_size, layers, bidirectional=True, batch_first=True)

        self.l1 = nn.Linear(hidden_size * 2, hidden_size)
        self.l1_norm = nn.LayerNorm(hidden_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # output will have shape
        # (batch, seq_length, hidden_size * 2 if bidireccional else hidden_size)
        # hidden will have shape 
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        outputs, hidden = self.gru(x)

        # In order to compact the forward and backward of the hidden state.
        # This tensor will pass through a Linear layer using the last state
        # of layers concatenated. It'll have shape
        # (batch, hidden_size * 2)
        # instead of the original shape of the hidden state
        # (layer * 2 if bidireccional else layer, batch_size, hidden_size)
        x = torch.concat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        x = self.l1(x)
        x = self.l1_norm(x)
        x = torch.tanh(x)

        # To match expected dimensions in gru decoder add 1 dimension along zero axis
        x = torch.unsqueeze(x, dim=0)

        # x para decoder hidden state
        # output para cross-attention
        return outputs, hidden, x


class DotProductAttention(nn.Module):
    
    def __init__(self, hidden_size):
        super().__init__()

        # Layer to align decoder/encoder hidden and encoder outputs
        # Encoder / Decoder produced hidden: (1, batch, hidden_size)
        # Encoder produced output: (batch, seq_len, hidden_size * 2)
        self.layer_compress_o = nn.Linear(hidden_size * 2, hidden_size)


    # Cross-Attention
    def forward(self, hidden, encoder_outputs):

        # Q -> hidde_state del Decoder (primera iteración hidden del encoder)
        # K y V -> output del Encoder

        Q = hidden.squeeze(0).unsqueeze(1)
        K = self.layer_compress_o(encoder_outputs)
        K_t = K.transpose(1, 2)

        d_k = Q.size(-1)
        energy_scaled = torch.bmm(Q, K_t) / (d_k ** 0.5)

        att_weights = torch.softmax(energy_scaled, dim=2)
        context = torch.bmm(att_weights, K)

        return att_weights, context.squeeze(1)


#class Decoder(nn.Module):
#    def __init__(self, vocab_size, embedding_size, hidden_size, layers, p_dropout):
#        super().__init__()

#        self.att = CrossAttention()

#        self.embedding = nn.Embedding(vocab_size, embedding_size)
#        self.dropout = nn.Dropout(p_dropout)

#        self.gru = nn.GRU(embedding_size, hidden_size, layers, batch_first=True)

#    def forward(self, x, decoder_hidden):
#        
#        x = self.embedding(x)
#        x = self.dropout(x)

#        outputs, hidden_state = self.gru(x, decoder_hidden)

#        return outputs, hidden_state, x

In [221]:
eng_batch, spa_o_batch, spa_t_batch = next(iter(train_loader))

encoder = Encoder(
    vocab_size=len(vocab_idx2word_eng), 
    embedding_size=300, 
    hidden_size=50,
    layers=1, 
    p_dropout=0.3
)
outputs_e, hidden_e, x_e = encoder(eng_batch)

print(f"Batch: {eng_batch.shape}")
print(f"Output_enc: {outputs_e.shape}")
print(f"Hidden_enc: {hidden_e.shape}")
print(f"Hidden_last_enc: {x_e.shape}")

dot_prod_att = DotProductAttention(x_e.size(-1))
att, con = dot_prod_att(x_e, outputs_e)

print("")
print(f"Att Weights: {att.shape}")
print(f"Context: {con.shape}")

Batch: torch.Size([64, 11])
Output_enc: torch.Size([64, 11, 100])
Hidden_enc: torch.Size([2, 64, 50])
Hidden_last_enc: torch.Size([1, 64, 50])

Att Weights: torch.Size([64, 1, 11])
Context: torch.Size([64, 50])
